1: RDFPaser

In [ ]:
import re, gzip

class RDFParser: 
    def __init__(self, file_path):
        self.file_path = file_path
        # Permissive N-Quads/N-Triples pattern
        self.counter_NM = 0
        self.pattern = re.compile(r"""
            ^\s*
            (?P<s>(<[^>]*>|_:[^\s]+))            # subject: IRI or blank node
            \s+
            (?P<p><[^>]*>)                       # predicate: IRI
            \s+
            (?P<o>
                "(?:[^"\\]|\\.)*"               # literal (handles escapes)
                (?:                              # optional language tag(s)
                    @[A-Za-z0-9_][A-Za-z0-9_-]* # allow underscore + hyphen
                    (?:@[A-Za-z0-9_][A-Za-z0-9_-]*)*
                )?
                (?:\^\^<[^>]*>)?                # optional datatype
              |
                <[^>]*>                         # OR IRI object
              |
                _:[^\s]+                        # OR blank node object
            )
            (?:\s+(?P<g>(<[^>]*>|_:[^\s]+)))?   # optional graph term
            \s*\.\s*$
        """, re.UNICODE | re.VERBOSE)

    def parse(self):
        with gzip.open(self.file_path, 'rt', encoding='utf-8', errors='replace') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                m = self.pattern.match(line)
                if m:
                    s = m.group('s')
                    p = m.group('p')
                    o = m.group('o')
                    g = m.group('g') if m.group('g') else None
                    yield (s, p, o, g)
                else:
                    self.counter_NM += 1
                    print("No match found", line)
        if self.counter_NM > 0:
            print(f"[RDFParser]Total non-matching lines: {self.counter_NM}")


2: Hashing 

In [ ]:
import hashlib

# Function to hash a string to a numeric ID
#vertex Id gen
def hash_id(s: str) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16)


3: Graph builder

In [ ]:
# The module
import networkx as nx
from hashing import hash_id

# Constants for RDF predicates for class signal
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"


#constructor with 3 switches 
class RDFGraphBuilder:
    def __init__(self, include_literals=False, use_numeric_ids=True, add_reverse_edges=False):
        #include_literals: bool = False = skip edges to literal objs
        #TD : literal should be collapced : as one big vertex
        #use_numeric_ids: bool = True = use numeric IDs for nodes saves memory
        #add_reverse_edges: bool = True = add reverse copy of each none-literal edges can add memory overhead
        #put the default on false
        self.include_literals = include_literals
        self.use_numeric_ids = use_numeric_ids
        self.add_reverse_edges = add_reverse_edges
        self.G = nx.MultiDiGraph()

    def _node_id(self, term: str):
        #if numeric id is true hashed int otherwise return the term as is
        return hash_id(term) if self.use_numeric_ids else term

    def _is_literal(self, term: str) -> bool:
        # Check if the term is a literal by looking for quotes
        return term.startswith('"')
    
    # def _is_blank_node(self, term: str) -> bool:
    #     #check if a node is a blank node
    #     return term.startswith("_:")

    def add_statement(self, s: str, p: str, o: str, g: str):
        #types to vertices not edges
        if p == RDF_TYPE and not self._is_literal(o):
            sid = self._node_id(s)
            self.G.add_node(sid, label=s)
            tset = self.G.nodes[sid].setdefault("types", set())
            tset.add(o)  # store the class URI (with <>)
            #print(f"Added type {o} to node {s}")
            return
        #if we decide include_literals is false this skips literals obj
        if self._is_literal(o) and not self.include_literals:
            return
        #adds the predicate and context on each edge
        sid = self._node_id(s)
        oid = self._node_id(o)
        self.G.add_node(sid, label=s)
        self.G.add_node(oid, label=o)
        self.G.add_edge(sid, oid, predicate=p, graph=g)
        #if reverse edges are enabled, add a reverse edge for non-literal objects
        #graph in both directions
        #but can increase memory usage significantly
        #only adds if the object is not a literal
        #and if add_reverse_edges is true
        if self.add_reverse_edges and not self._is_literal(o):
            self.G.add_edge(oid, sid, predicate=p, graph=g, _reversed=True)

    def build(self, triple_iterable):
        for s, p, o, g in triple_iterable:
            self.add_statement(s, p, o, g)
        return self.G


4: Vertexsummary

In [ ]:
from __future__ import annotations

from collections import defaultdict
from typing import Dict, Set, Iterable, Optional
from hashing import hash_id


class VertexSummary:
    #FLUID-style summarization
    #types: rdf:type class IRIs of the source vertex
    #neighbors: predicate--count of outgoing uses
    #neighbor_dst_types: predicate -- set of destination class IRIs (dst rdf:types)
    #payload:set of named graph IRIs(sources)
    #instances:set of subject IRIs this summary represents
    #labels:for debug can hold human labels or class IRIs

    # Typ signature (get_id):
    #   (types, set(predicates), per-predicate dst type sets)

    # constructor
    def __init__(self, types: Optional[Iterable[str]] = None):
        self.types: Set[str] = set(types or [])
        self.neighbors: Dict[str, int] = defaultdict(int)
        self.neighbor_dst_types: Dict[str, Set[str]] = defaultdict(set)
        self.payload: Set[str] = set()
        self.instances: Set[str] = set()
        self.labels: Set[str] = set()  # optional; you may ignore if unused

    #adds rdf:type class to the summary
    def add_type(self, t: Optional[str]) -> None:
        if t:
            self.types.add(t)

    def add_types(self, ts: Iterable[str]) -> None:
        for t in ts:
            if t:
                self.types.add(t)
    #neighbors increase the count and update the set
    def add_neighbor(self, predicate_iri: Optional[str]) -> None:
        if predicate_iri:
            self.neighbors[predicate_iri] += 1
    #for the given predicate (union in dst type)
    def add_dst_types(self, predicate_iri: Optional[str], dst_types: Iterable[str]) -> None:
        if not predicate_iri:
            return
        bucket = self.neighbor_dst_types[predicate_iri]
        for t in dst_types:
            if t:
                bucket.add(t)
    #record source
    def add_payload(self, graph_iri: Optional[str]) -> None:
        if graph_iri:
            self.payload.add(graph_iri)
    #subject IRI that this summary blongs to
    def add_instance(self, subject_iri: Optional[str]) -> None:
        if subject_iri:
            self.instances.add(subject_iri)
    #opt human readable names
    def add_label(self, label: Optional[str]) -> None:
        if label:
            self.labels.add(label)

    #merge -> Scala-mergeMessage
    #merges 2 summarias and updates on vertex -> combines sets inside one summary
    def merge(self, other: "VertexSummary") -> "VertexSummary":
        #Union of sets and sum neighbor predicate counts 
        #union "per predicat" of neighbor_dst_types
        if other is self:
            return self
        self.types |= other.types
        self.payload |= other.payload
        self.instances |= other.instances
        self.labels |= other.labels
        for p, c in other.neighbors.items():
            self.neighbors[p] += c
        for p, dsts in other.neighbor_dst_types.items():
            self.neighbor_dst_types[p].update(dsts)
        return self

    #signature / ID
    def get_id(self) -> int:
        # build signature string and hash it
        # signature components:
        # 1-sorted(types)
        # 2-sorted(outgoing predicate keys)
        # 3-for each predicate (sorted by name), the sorted dst type set
        parts: list[str] = []

        # 1 types
        if self.types:
            parts.extend(sorted(self.types))
        parts.append("|#|")

        # 2 outgoing predicates (set)
        if self.neighbors:
            parts.extend(sorted(self.neighbors.keys()))
        parts.append("|@|")

        # 3 predicate -> dst type sets (schema-aware behavior)
        if self.neighbor_dst_types:
            for p in sorted(self.neighbor_dst_types.keys()):
                parts.append(p)
                parts.extend(sorted(self.neighbor_dst_types[p]))

        signature = "\u241F".join(parts)  # reduce collisions
        return hash_id(signature)

    # export
    def as_dict(self) -> Dict[str, object]:
        return {
            "types": sorted(self.types),
            "predicates": sorted(self.neighbors.keys()),
            "predicate_counts": dict(self.neighbors),
            "predicate_to_dst_types": {p: sorted(ts) for p, ts in self.neighbor_dst_types.items()},
            "payload_graphs": sorted(self.payload),
            "instances": sorted(self.instances),
            "labels": sorted(self.labels),
            "id": self.get_id(),
        }

    def __repr__(self) -> str:
        return f"VertexSummary(id={self.get_id()}, types={len(self.types)}, preds={len(self.neighbors)})"


5: SchemaEtraction

In [ ]:
from __future__ import annotations
import networkx as nx
from typing import Protocol #define flexible interfaces
from VertexSummary import VertexSummary

class SchemaExtraction(Protocol):
    # Scala trait analogue:
    #   - send_message: build a per-edge summary (for the source vertex).
    #   - merge_messages: combine two summaries into one (default = union).

    def send_message(self, G: nx.MultiDiGraph, src, dst, edata: dict) -> VertexSummary:
        #VertexSummary from ONE edge (src->edata()'predicate')-> dst)
        # Parameters
        # G: NetworkX MultiDiGraph(built RDF graph)
        # src: source node id (hash int or string URI)
        # dst: destination node id
        # edata: edge attribute dict-has:
        #  'predicate': str (IRI in <>)
        #  'graph'    : str (named graph IRI in <>)

        # Returns
        # VertexSummary describing the source vertex contribution of the specific edge.
        raise NotImplementedError

    def merge_messages(self, a: VertexSummary, b: VertexSummary) -> VertexSummary:
        # 2 summaries with the same vertex -> new merged summary and can be strategized
        # merge two partial summaries for the same vertex into one.
        # default behavior is set-union + count-sum, so delegate to VertexSummary.merge.
        return a.merge(b)


6: Summarizer

In [ ]:
from collections import defaultdict
from typing import Dict, List, Tuple, Iterable, Any
import networkx as nx


def group_by_signature(per_vertex_id: Dict[Any, int]) -> Dict[int, List[Any]]:
    # Group raw graph nodes into FLUID supernodes by their summary signature id.
    # Parameters
    # per_vertex_id : dict[node] -> signature_id

    # Returns
    # groups : dict[signature_id] -> list[node]
    groups: Dict[int, List[Any]] = defaultdict(list)
    for node, sid in per_vertex_id.items():
        groups[sid].append(node)
    return groups


def build_summary_graph(
    G: nx.MultiDiGraph,
    groups: Dict[int, List[Any]],
    *,
    by_predicate: bool = True,
) -> Tuple[Dict[int, Dict[int, int]], Dict[int, Dict[int, Dict[str, int]]]]:
    # Build a summary (supernode) edge view from the raw graph.

    # Parameters
    # G: NetworkX MultiDiGraph
    # groups: dict[signature_id] -> list[node]
    # by_predicate: if True, also count edges per predicate label

    # Returns
    # super_counts: dict[src_sid][dst_sid] = total edge count between supernodes
    # super_pred: dict[src_sid][dst_sid][predicate] = count per predicate (if by_predicate=True)

    # Invert groups: raw node -> supernode id
    node2sid: Dict[Any, int] = {}
    for sid, members in groups.items():
        for n in members:
            node2sid[n] = sid

    super_counts: Dict[int, Dict[int, int]] = defaultdict(lambda: defaultdict(int))
    super_pred: Dict[int, Dict[int, Dict[str, int]]] = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

    # Walk raw edges and accumulate into superedges
    for u, v, data in G.edges(data=True):
        su = node2sid.get(u)
        sv = node2sid.get(v)
        if su is None or sv is None:
            continue
        super_counts[su][sv] += 1
        if by_predicate:
            p = data.get("predicate")
            if p is not None:
                super_pred[su][sv][p] += 1

    return super_counts, super_pred


7: Collector

In [ ]:
# collector.py
from typing import Dict, Tuple
import networkx as nx
from VertexSummary import VertexSummary  # class with merge() and get_id()
from SchemaExtraction import SchemaExtraction  # Protocol or base class


class SchemaCollector:
    # GraphX simulator for python:
    # walks all edges (u -> v) in G
    # uses `strategy.send_message(...)` to build a per-edge VertexSummary
    # merges messages per SOURCE vertex using `strategy.merge_messages(...)`
    # returns:
    # per_vertex_summary: node -> merged VertexSummary
    # per_vertex_id:      node -> signature id (for FLUID grouping)
    def __init__(self, strategy: SchemaExtraction):
        self.strategy = strategy

    def collect(self, G: nx.MultiDiGraph) -> Tuple[Dict[object, VertexSummary], Dict[object, int]]:
        mailboxes: Dict[object, list[VertexSummary]] = {}

        # 1) send (GraphX sendMessage)
        for u, v, edata in G.edges(data=True):
            msg = self.strategy.send_message(G, u, v, edata)
            mailboxes.setdefault(u, []).append(msg)

        # 2) merge (GraphX mergeMessage)
        per_vertex_summary: Dict[object, VertexSummary] = {}
        per_vertex_id: Dict[object, int] = {}

        for node in G.nodes:
            msgs = mailboxes.get(node, [])
            merged = VertexSummary(types=set(G.nodes[node].get("types", [])))
            merged.add_instance(G.nodes[node].get("label"))
            for m in msgs:
                merged = self.strategy.merge_messages(merged, m)
            per_vertex_summary[node] = merged
            per_vertex_id[node] = merged.get_id()

        return per_vertex_summary, per_vertex_id


8: Attribute collection

In [ ]:
import networkx as nx
from VertexSummary import VertexSummary
from SchemaExtraction import SchemaExtraction

class SEAttributeCollection(SchemaExtraction):
    # Attribute-only strategy
    # take src types (from node attrs)
    # record the predicate used on this edge
    # record the graph
    # record the src instance (its label/IRI)
    # """
    def send_message(self, G: nx.MultiDiGraph, src, dst, edata: dict) -> VertexSummary:
        src_types = set(G.nodes[src].get("types", []))
        vs = VertexSummary(types=src_types)
        vs.add_instance(G.nodes[src].get("label"))
        vs.add_neighbor(edata.get("predicate"))
        vs.add_payload(edata.get("graph"))
        return vs


9: Class Collection

In [ ]:
# SEClassCollection.py
import networkx as nx
from VertexSummary import VertexSummary
from SchemaExtraction import SchemaExtraction

class SEClassCollection(SchemaExtraction):
    # Class-only collection strategy (Scala SE_ClassCollection analogue)

    def send_message(self, G: nx.MultiDiGraph, src, dst, edata: dict) -> VertexSummary:
        # Start a new summary element
        vs = VertexSummary()

        # Collect origin (src) types
        src_types = set(G.nodes[src].get("types", []))
        vs.add_types(src_types)

        # Add payload: which named graph (dataset source)
        vs.add_payload(edata.get("graph"))

        # Add instance: the actual subject URI
        vs.add_instance(G.nodes[src].get("label"))

        return vs

    # merge_messages just defaults to VertexSummary.merge via SchemaExtraction


10: Attributeclass Collection

In [ ]:
# SEComplexAttributeClassCollection.py
import networkx as nx
from VertexSummary import VertexSummary
from SchemaExtraction import SchemaExtraction

class SEComplexAttributeClassCollection(SchemaExtraction):
    # Complex Attribute+Class strategy (Scala SE_ComplexAttributeClassCollection analogue)

    def send_message(self, G: nx.MultiDiGraph, src, dst, edata: dict) -> VertexSummary:
        vs_src = VertexSummary()

        # Get origin (src) types
        src_types = set(G.nodes[src].get("types", []))
        vs_src.add_types(src_types)

        # Get destination (dst) types
        dst_types = set(G.nodes[dst].get("types", []))

        # Add neighbor predicate
        predicate = edata.get("predicate")
        vs_src.add_neighbor(predicate)

        # Attach destination's types to this predicate
        vs_src.add_dst_types(predicate, dst_types)

        # Add payload (graph IRI)
        vs_src.add_payload(edata.get("graph"))

        # Add instance (the actual subject URI)
        vs_src.add_instance(G.nodes[src].get("label"))

        return vs_src
    # merge_messages just defaults to VertexSummary.merge via SchemaExtraction

11:Save

In [ ]:
import os, json
from typing import Dict, Any, Union, Type
import networkx as nx

#  project dependencies ===
from Collector import SchemaCollector
from AttCollection import SEAttributeCollection
from ClassCollection import SEClassCollection
from CompAttClassCollection import SEComplexAttributeClassCollection
from Summarizer import group_by_signature, build_summary_graph

def _sorted_list(x) -> list:
    # Convert any iterable (e.g., set) into a sorted list for deterministic JSON.
    return sorted(list(x))

def _canon_pred2dst(pred2dst: Dict[str, set]) -> Dict[str, list]:
    # {predicate -> set(dst_types)} into
    # {predicate -> sorted list(dst_types)}, with predicates sorted too.
    
    return {p: _sorted_list(v) for p, v in sorted(pred2dst.items())}

# Registry of strategy names -> classes
_STRATEGY_REGISTRY: Dict[str, Type] = {
    "attribute": SEAttributeCollection,
    "class": SEClassCollection,
    "attributeclass": SEComplexAttributeClassCollection,
}

#save summary for a specific graph for a specific strategy 
#path + metadata
#save a canonical JSON package (schema + payload + edges + membership)
def save_summary_for_strategy(
    G: nx.MultiDiGraph,
    strategy: Union[str, Type, object] = "attributeclass",  
    out_path: str = "summaries/summary.json",
    *,
    meta: Dict[str, Any] = None,
    include_membership: bool = False,
) -> Dict[str, Any]:

    # 1) Get strategy instance
    if isinstance(strategy, str):
        key = strategy.strip().lower()
        if key not in _STRATEGY_REGISTRY:
            raise ValueError(f"Unknown strategy '{strategy}', choose one of {list(_STRATEGY_REGISTRY)}")
        strat = _STRATEGY_REGISTRY[key]()
        strategy_name = key
    elif isinstance(strategy, type):
        strat = strategy()
        strategy_name = strategy.__name__
    else:
        strat = strategy
        strategy_name = type(strat).__name__

    # 2) Collect per-vertex summaries with this strategy
    collector = SchemaCollector(strat)
    per_v_summary, per_v_id = collector.collect(G)

    # 3) Group nodes and build superedge counts
    groups = group_by_signature(per_v_id)
    super_counts, super_pred = build_summary_graph(G, groups, by_predicate=True)

    # 4) make into JSON-serializable structure
    
    nodes, edges, membership = {}, {}, {}
    for sid, members in groups.items():
        types, preds, dsp = set(), set(), set()
        pred2dst: Dict[str, set] = {}
        for n in members:
            vs = per_v_summary.get(n)
            if not vs:
                continue
            types |= set(vs.types)
            preds |= set(vs.neighbors.keys())
            dsp |= set(vs.payload)
            for p, dsts in vs.neighbor_dst_types.items():
                pred2dst.setdefault(p, set()).update(dsts)
        nodes[str(sid)] = {
            "sid": int(sid),
            "vcp": len(members),                          # payload: vertex count
            "dsp": _sorted_list(dsp),                     # payload: data sources
            "types": _sorted_list(types),                 # schema: rdf:types
            "predicates": _sorted_list(preds),            # schema: property set
            "pred_to_dst_types": _canon_pred2dst(pred2dst) # schema: predicate→dst types
        }

    for su, dsts in super_counts.items():
        for sv, total in dsts.items():
            pred_counts = {
                p: int(c) for p, c in sorted(
                    super_pred.get(su, {}).get(sv, {}).items()
                )
            }
            edges[f"{su}->{sv}"] = {
                "src": int(su),
                "dst": int(sv),
                "count": int(total),       #payload: rolled edge count
                "pred_counts": pred_counts #payload: histogram per predicate
            }

    pkg = {
        "meta": {"strategy": strategy_name, **(meta or {})},
        "nodes": dict(sorted(nodes.items())),
        "edges": dict(sorted(edges.items())),
    }

    if include_membership:
        membership = {}
        for node, sid in per_v_id.items():
            label = G.nodes[node].get("label", node) if node in G.nodes else str(node)
            membership[str(label)] = int(sid)
        pkg["membership"] = dict(sorted(membership.items()))

    # 5) Save JSON
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(pkg, f, ensure_ascii=False, indent=2, sort_keys=True)

    print(f"[save] wrote summary ({strategy_name}) → {out_path}")
    return pkg

def load_summary_package(path: str) -> Dict[str, Any]:
    #Load a previously saved summary JSON package back into a dict
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

A test on a small sub graph

In [ ]:
# Test AttributeCollection on a small subgraph
file_path = "2012-05-06/data-seedlist.nq.gz"  # path to your RDF data file
# --- Step 1: Parse only a handful of triples ---
parser = RDFParser(file_path, max_triples=50000)   # just 50 triples for testing
builder = RDFGraphBuilder(
    include_literals=False,
    use_numeric_ids=False,
    add_reverse_edges=False
)
G = builder.build(parser.parse())

print(f"[Test] Built small graph with {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# --- Step 2: Collect summaries using AttributeCollection ---
strategy = SEAttributeCollection()
collector = SchemaCollector(strategy)
per_vertex_summary, per_vertex_id = collector.collect(G)

# --- Step 3: Group into FLUID supernodes ---
groups = group_by_signature(per_vertex_id)

print(f"[Test] Found {len(groups)} unique signature groups")
for sid, members in list(groups.items())[:5]:  # show first 5 groups
    print(f"Supernode {sid} with {len(members)} members -> sample: {members[:3]}")

# --- Step 4: Build summary graph ---
super_counts, super_pred = build_summary_graph(G, groups)

print(f"[Test] Summary graph has {len(super_counts)} supernodes")
for su, dsts in list(super_counts.items())[:3]:
    for sv, count in dsts.items():
        preds = super_pred[su][sv]
        print(f"  Supernode {su} -> {sv} : {count} edges ({dict(preds)})")


Added type <http://www.w3.org/2000/01/rdf-schema#Class> to node <http://rdf.data-vocabulary.org/#Person>
Added type <http://xmlns.com/foaf/0.1/PersonalProfileDocument> to node <http://foaf.me/Djing>
Added type <http://www.w3.org/2002/07/owl#AnnotationProperty> to node <http://www.w3.org/2001/vcard-rdf/3.0#ADR>
Added type <http://purl.org/rss/1.0/channel> to node <http://captsolo.net/info/blog_a.php>
Added type <http://kent.zpr.fer.hr:8080/educationalProgram/vocab/sisvu.rdf#Academy> to node <http://kent.zpr.fer.hr:8080/educationalProgram/resource/organization/314>
Added type <http://xmlns.com/foaf/0.1/Person> to node <http://gromgull.net/me>
Added type <http://xmlns.com/foaf/0.1/Person> to node <http://sebastian.tramp.name>
Added type <http://xmlns.com/foaf/0.1/Person> to node <http://mamund.com/foaf.rdf#me>
Added type <http://www.w3.org/2002/07/owl#Class> to node _:httpx3Ax2Fx2Fswx2Dportalx2Ederix2Eorgx2Fontologiesx2FswportalxxA0
Added type <http://xmlns.com/foaf/0.1/Document> to node 

Unit tests for all strategy

In [12]:
# === TEST CELL: Section-by-section verification for the notebook pipeline ===
# RDFParser, RDFGraphBuilder, VertexSummary, SEAttributeCollection, SEClassCollection,
# SEComplexAttributeClassCollection, SchemaCollector, group_by_signature, build_summary_graph

import os, gzip, tempfile, traceback

RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"

NQ_SAMPLE = f"""\
<tbl> {RDF_TYPE} <Person> <http://zbw.eu> .
<tbl> <name> "Till Blume" <http://zbw.eu> .
<tbl> <worksFor> <ZBW> <http://zbw.eu> .
<tbl> <livesIn> "Kiel" <http://zbw.eu> .
<ZBW> {RDF_TYPE} <Organisation> <http://zbw.eu> .
<ZBW> <name> "Leibniz Information Centre for Economics" <http://zbw.eu> .
"""

def _write_temp_nq_gz(content: str) -> str:
    fd, path = tempfile.mkstemp(suffix=".nq.gz", prefix="mini-", text=False)
    os.close(fd)
    with gzip.open(path, "wb") as f:
        f.write(content.encode("utf-8"))
    return path

def _run(test_fn):
    name = test_fn.__name__
    try:
        test_fn()
        print(f" {name} PASSED")
    except AssertionError as e:
        print(f" {name} FAILED: {e}")
        traceback.print_exc(limit=1)
    except Exception as e:
        print(f" {name} ERROR: {e}")
        traceback.print_exc(limit=1)

# 1) RDFParser.parse()
# -------------------------------
def test_1_rdfparser_reads_gz_and_max_triples():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        parser_full = RDFParser(path)
        triples_full = list(parser_full.parse())
        assert len(triples_full) == 6 #Expected 6 triples from sample

        parser_cut = RDFParser(path, max_triples=2)
        triples_cut = list(parser_cut.parse())
        assert len(triples_cut) == 2 #max_triples=2 should stop after 2 items

        # tuple shape: (s, p, o, g)
        s, p, o, g = triples_full[0]
        assert s.startswith("<") and p.startswith("<") and g.startswith("<") #N-Quads parsing shape mismatch
    finally:
        os.remove(path)

In [13]:
def test_2_graph_builder_types_and_edges_directed():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        G = RDFGraphBuilder(include_literals=True, use_numeric_ids=False, add_reverse_edges=False)\
            .build(RDFParser(path).parse())

        # Nodes and edges exist
        assert G.number_of_nodes() >= 2 #Graph should have at least 2 nodes
        # Literal edges are skipped; only <tbl> -> <ZBW> (worksFor) is non-literal
        assert G.number_of_edges() >= 1 #Expected at least one non-literal edge (worksFor)

        # types are stored on nodes
        by_label = {G.nodes[n].get("label"): n for n in G.nodes}
        assert "<tbl>" in by_label and "<ZBW>" in by_label #Expected <tbl> and <ZBW> nodes to exist

        tbl = by_label["<tbl>"]
        zbw = by_label["<ZBW>"]
        assert "<Person>" in G.nodes[tbl].get("types", set()) #tbl should have type <Person>
        assert "<Organisation>" in G.nodes[zbw].get("types", set()) #ZBW should have type <Organisation>

        # edge attributes include 'predicate'
        u, v, d = next(iter(G.edges(data=True)))
        assert "predicate" in d and isinstance(d["predicate"], str) #Edge must carry a 'predicate' str
    finally:
        os.remove(path)

def test_2b_graph_builder_reverse_edges_flag():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        G = RDFGraphBuilder(include_literals=False, use_numeric_ids=False, add_reverse_edges=True)\
            .build(RDFParser(path).parse())
        # With reverse edges on, at least the forward + reverse for worksFor
        assert G.number_of_edges() >= 2 #Reverse edges should at least double the non-literal relation
        # Confirm a reverse edge is marked if your builder sets _reversed=True
        any_marked = any(d.get("_reversed") for _,_,d in G.edges(data=True))
        # This flag is optional; don't fail if not present—just print what we found
        print(f"    [info] reverse-edge flag present on any edge: {any_marked}")
    finally:
        os.remove(path)

In [14]:
def test_3_attribute_collection_and_collect():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        G = RDFGraphBuilder(include_literals=False, use_numeric_ids=False, add_reverse_edges=False)\
            .build(RDFParser(path).parse())

        strategy = SEAttributeCollection()
        collector = SchemaCollector(strategy)
        per_vertex_summary, per_vertex_id = collector.collect(G)

        assert isinstance(per_vertex_summary, dict) and isinstance(per_vertex_id, dict)
        assert len(per_vertex_summary) >= 1 #Expected at least one vertex summary

        # Ensure 'worksFor' shows up in at least one summary (predicate recorded)
        assert any("<worksFor>" in vs.neighbors for vs in per_vertex_summary.values()) #Expected '<worksFor>' in attribute neighbors

        # payload and instance recorded
        assert any(vs.payload for vs in per_vertex_summary.values()) #Expected payload (graph IRIs)
        assert any(vs.instances for vs in per_vertex_summary.values()) #Expected instances (subject IRIs)

        # signature ids are ints
        assert all(isinstance(sid, int) for sid in per_vertex_id.values()) #Signature IDs must be ints
    finally:
        os.remove(path)

# ----------------------------------------------------
# 4) SEClassCollection + SchemaCollector (Cell 8)
# ----------------------------------------------------
def test_4_class_collection_and_collect():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        G = RDFGraphBuilder(include_literals=False, use_numeric_ids=False, add_reverse_edges=False)\
            .build(RDFParser(path).parse())

        strategy = SEClassCollection()
        collector = SchemaCollector(strategy)
        per_vertex_summary, per_vertex_id = collector.collect(G)

        # Expect class info to be aggregated
        types_union = set()
        instances_union = set()
        payload_union = set()
        for vs in per_vertex_summary.values():
            types_union |= vs.types
            instances_union |= vs.instances
            payload_union |= vs.payload

        assert "<Person>" in types_union and ("<Organisation>" in types_union or "<Organization>" in types_union)
        #Expected Person and Organisation types in class-only summaries
        assert "<tbl>" in instances_union and "<ZBW>" in instances_union #Expected instances recorded
        assert len(payload_union) >= 1 #Expected payload (graph IRIs)
    finally:
        os.remove(path)


In [15]:
def test_5_complex_attribute_class_and_collect():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        G = RDFGraphBuilder(include_literals=False, use_numeric_ids=False, add_reverse_edges=False)\
            .build(RDFParser(path).parse())

        strategy = SEComplexAttributeClassCollection()
        collector = SchemaCollector(strategy)
        per_vertex_summary, per_vertex_id = collector.collect(G)

        # Check predicates and dst-type associations are captured
        has_predicates = any(vs.neighbors for vs in per_vertex_summary.values())
        has_dst_types = any(any(len(ts) > 0 for ts in vs.neighbor_dst_types.values())
                            for vs in per_vertex_summary.values())
        assert has_predicates #Expected predicate counts in complex strategy
        assert has_dst_types #Expected predicate→destination-types mapping in complex strategy
    finally:
        os.remove(path)

In [17]:
def test_6_grouping_and_summary_graph_helpers():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        G = RDFGraphBuilder(include_literals=False, use_numeric_ids=False, add_reverse_edges=False)\
            .build(RDFParser(path).parse())

        # Use AttributeCollection as in your main test cell
        strategy = SEAttributeCollection()
        collector = SchemaCollector(strategy)
        per_vertex_summary, per_vertex_id = collector.collect(G)

        # Group raw nodes into supernodes by signature
        groups = group_by_signature(per_vertex_id)
        assert isinstance(groups, dict) and len(groups) >= 1 #Expected ≥1 signature group

        # Build summary (supernode) edge view
        super_counts, super_pred = build_summary_graph(G, groups, by_predicate=True)
        # At least one super-edge should exist
        any_super_edge = any(dsts for dsts in super_counts.values())
        assert any_super_edge #Expected at least one super-edge

        # Predicate histograms should be dicts of counts
        for su, dsts in super_pred.items():
            for sv, pcounts in dsts.items():
                assert isinstance(pcounts, dict)
                for p, c in pcounts.items():
                    assert isinstance(p, str) and isinstance(c, int) and c >= 1
    finally:
        os.remove(path)

In [18]:
def test_7_end_to_end_attribute_flow():
    path = _write_temp_nq_gz(NQ_SAMPLE)
    try:
        # Build
        builder = RDFGraphBuilder(include_literals=False, use_numeric_ids=False, add_reverse_edges=False)
        G = builder.build(RDFParser(path).parse())
        assert G.number_of_nodes() >= 2 and G.number_of_edges() >= 1

        # Collect
        strategy = SEAttributeCollection()
        per_vertex_summary, per_vertex_id = SchemaCollector(strategy).collect(G)

        # Group
        groups = group_by_signature(per_vertex_id)
        assert len(groups) >= 1

        # Summarize edges
        super_counts, super_pred = build_summary_graph(G, groups, by_predicate=True)
        # Sanity: super_pred predicates should include "<worksFor>" somewhere
        found_worksfor = any("<worksFor>" in pcounts for dsts in super_pred.values() for pcounts in dsts.values())
        assert found_worksfor #Expected <worksFor> to appear in superedge predicate counts
    finally:
        os.remove(path)

In [19]:
_run(test_1_rdfparser_reads_gz_and_max_triples)
_run(test_2_graph_builder_types_and_edges_directed)
_run(test_2b_graph_builder_reverse_edges_flag)
_run(test_3_attribute_collection_and_collect)
_run(test_4_class_collection_and_collect)
_run(test_5_complex_attribute_class_and_collect)
_run(test_6_grouping_and_summary_graph_helpers)
_run(test_7_end_to_end_attribute_flow)

print("=== Test cell finished ===")

[Parser] Stopping early at 2 triples
 test_1_rdfparser_reads_gz_and_max_triples PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
 test_2_graph_builder_types_and_edges_directed PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
    [info] reverse-edge flag present on any edge: True
 test_2b_graph_builder_reverse_edges_flag PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
 test_3_attribute_collection_and_collect PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
 test_4_class_collection_and_collect PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
 test_5_complex_attribute_class_and_collect PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
 test_6_grouping_and_summary_graph_helpers PASSED
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
 test_7_end_to_end_attribute_flow PASSED
===

In [28]:
# === GOLDEN STANDARD TESTS for VertexSummary (parity with Scala VertexSummaryTest) ===
from VertexSummary import VertexSummary
import traceback

def _run(fn):
    try:
        fn()
        print(f"{fn.__name__} PASSED")
    except AssertionError as e:
        print(f"{fn.__name__} FAILED: {e}")
        traceback.print_exc(limit=1)
    except Exception as e:
        print(f"{fn.__name__} ERROR: {e}")
        traceback.print_exc(limit=1)

# Two empty summaries → same ID
def test_empty_schema_element():
    s1 = VertexSummary()
    s2 = VertexSummary()
    assert s1.get_id() == s2.get_id() #Empty VertexSummary objects should have identical IDs

# Neighbor insertion order should not matter
def test_order_schema_element():
    se1 = VertexSummary()
    se1.add_neighbor("worksfor")
    se1.add_neighbor("name")
    se1.add_neighbor("livesin")

    se2 = VertexSummary()
    se2.add_neighbor("livesin")
    se2.add_neighbor("name")
    se2.add_neighbor("worksfor")

    assert se1.get_id() == se2.get_id() #Neighbor insertion order must not affect signature ID

# Payload differences must not affect ID
def test_payload_for_schema():
    se1 = VertexSummary()
    for p in ["worksfor", "name", "livesin"]:
        se1.add_neighbor(p)
    se1.add_payload("payloadA")

    se2 = VertexSummary()
    for p in ["worksfor", "name", "livesin"]:
        se2.add_neighbor(p)
    se2.add_payload("payloadB")

    assert se1.get_id() == se2.get_id() 

# Instance differences must not affect ID
def test_summarized_instance_for_schema():
    se1 = VertexSummary()
    for p in ["worksfor", "name", "livesin"]:
        se1.add_neighbor(p)
    se1.add_instance("instanceA")

    se2 = VertexSummary()
    for p in ["worksfor", "name", "livesin"]:
        se2.add_neighbor(p)
    se2.add_instance("instanceB")

    assert se1.get_id() == se2.get_id()

_run(test_empty_schema_element)
_run(test_order_schema_element)
_run(test_payload_for_schema)
_run(test_summarized_instance_for_schema)

test_empty_schema_element PASSED
test_order_schema_element PASSED
test_payload_for_schema PASSED
test_summarized_instance_for_schema PASSED


In [20]:
# DATA-DRIVEN MINI HARNESS for testing many tiny unit graphs 

import os, gzip, tempfile, traceback
from typing import Dict, Iterable, Optional, Type

# exact rdf:type IRI your builder expects
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"

def _write_temp_nq_gz(content: str) -> str:
    fd, path = tempfile.mkstemp(suffix=".nq.gz", prefix="unit-", text=False)
    os.close(fd)
    with gzip.open(path, "wb") as f:
        f.write(content.encode("utf-8"))
    return path

def run_unit_graph_test(
    nq_text: str,
    *,
    include_literals: bool = False,
    use_numeric_ids: bool = False,
    add_reverse_edges: bool = False,
    strategy_cls: Optional[Type] = None,   # e.g., SEAttributeCollection
    expect_min_nodes: int = 1,
    expect_min_edges: int = 0,
    expect_types_on_label: Optional[Dict[str, Iterable[str]]] = None,   # {"<tbl>": ["<Person>"], ...}
    expect_predicates_present: Optional[Iterable[str]] = None,          # ["<worksFor>", ...]
    expect_superedge_predicates_present: Optional[Iterable[str]] = None # predicates to appear between supernodes
):
    # """
    # Build graph -> (optionally) collect schema with `strategy_cls` -> group -> build summary graph.
    # Checks only the expectations you provide (others are skipped).
    # """
    path = _write_temp_nq_gz(nq_text)
    try:
        # 1) Build graph
        parser = RDFParser(path)
        builder = RDFGraphBuilder(
            include_literals=include_literals,
            use_numeric_ids=use_numeric_ids,
            add_reverse_edges=add_reverse_edges
        )
        G = builder.build(parser.parse())

        # basic graph expectations
        assert G.number_of_nodes() >= expect_min_nodes, f"nodes {G.number_of_nodes()} < expected min {expect_min_nodes}"
        assert G.number_of_edges() >= expect_min_edges, f"edges {G.number_of_edges()} < expected min {expect_min_edges}"

        # check types on specific node labels
        if expect_types_on_label:
            by_label = {G.nodes[n].get("label"): n for n in G.nodes}
            for label, types_wanted in expect_types_on_label.items():
                assert label in by_label, f"Node with label {label} not found"
                n = by_label[label]
                ntypes = set(G.nodes[n].get("types", set()))
                missing = set(types_wanted) - ntypes
                assert not missing, f"{label} missing types: {missing} (has {ntypes})"

        per_vertex_summary = {}
        per_vertex_id = {}
        # 2)  run a schema strategy
        if strategy_cls is not None:
            strategy = strategy_cls()
            collector = SchemaCollector(strategy)
            per_vertex_summary, per_vertex_id = collector.collect(G)
            # if you asked for predicates in summaries, check them
            if expect_predicates_present:
                found = set()
                for vs in per_vertex_summary.values():
                    found |= set(vs.neighbors.keys())
                for p in expect_predicates_present:
                    assert p in found, f"predicate {p} not found in per-vertex summaries; found={found}"

        # 3) Group + 4) superedge view (only if we ran a strategy and have signatures)
        if per_vertex_id:
            groups = group_by_signature(per_vertex_id)
            super_counts, super_pred = build_summary_graph(G, groups, by_predicate=True)

            # check that certain predicates appear on superedges
            if expect_superedge_predicates_present:
                super_found = set()
                for su, dsts in super_pred.items():
                    for sv, pcounts in dsts.items():
                        super_found |= set(pcounts.keys())
                for p in expect_superedge_predicates_present:
                    assert p in super_found, f"superedge predicate {p} not found; found={super_found}"

        print("PASSED")
        return G, per_vertex_summary, per_vertex_id  # handy for inspection in the notebook
    except AssertionError as e:
        print(f"FAILED: {e}")
        traceback.print_exc(limit=1)
    except Exception as e:
        print(f"ERROR: {e}")
        traceback.print_exc(limit=1)
    finally:
        os.remove(path)

# ---- EXAMPLES: three tiny graphs to show how to reuse the same harness ----

# A) Your original 2-node graph (literals skipped, directed)
NQ_ORIG = f"""\
<tbl> {RDF_TYPE} <Person> <http://zbw.eu> .
<tbl> <name> "Till Blume" <http://zbw.eu> .
<tbl> <worksFor> <ZBW> <http://zbw.eu> .
<tbl> <livesIn> "Kiel" <http://zbw.eu> .
<ZBW> {RDF_TYPE} <Organisation> <http://zbw.eu> .
<ZBW> <name> "Leibniz Information Centre for Economics" <http://zbw.eu> .
"""

print("Graph A — original graph + AttributeCollection")
run_unit_graph_test(
    NQ_ORIG,
    include_literals=False,
    add_reverse_edges=False,
    strategy_cls=SEAttributeCollection,
    expect_min_nodes=2,
    expect_min_edges=1,  # only <worksFor> survives
    expect_types_on_label={"<tbl>": ["<Person>"], "<ZBW>": ["<Organisation>"]},
    expect_predicates_present=["<worksFor>"],
    expect_superedge_predicates_present=["<worksFor>"],
)

# B) Same graph but UNDIRECTED view (reverse edges added)
print("\nGraph B — original graph + reverse edges (undirected-ish)")
run_unit_graph_test(
    NQ_ORIG,
    include_literals=False,
    add_reverse_edges=True,
    strategy_cls=SEAttributeCollection,
    expect_min_nodes=2,
    expect_min_edges=2,  # forward + reverse for worksFor
    expect_predicates_present=["<worksFor>"],
    expect_superedge_predicates_present=["<worksFor>"],
)

# C) Graph with an extra non-literal relation and a second org (scales with same asserts)
NQ_EXTRA = f"""\
<tbl> {RDF_TYPE} <Person> <http://zbw.eu> .
<tbl> <worksFor> <ZBW> <http://zbw.eu> .
<tbl> <collabWith> <ORG2> <http://zbw.eu> .
<ZBW> {RDF_TYPE} <Organisation> <http://zbw.eu> .
<ORG2> {RDF_TYPE} <Organisation> <http://zbw.eu> .
"""
print("\nGraph C — extended graph (two non-literal edges, two orgs)")
run_unit_graph_test(
    NQ_EXTRA,
    include_literals=False,
    add_reverse_edges=False,
    strategy_cls=SEComplexAttributeClassCollection,  # try the complex one
    expect_min_nodes=3,
    expect_min_edges=2,  # worksFor + collabWith
    expect_types_on_label={"<tbl>": ["<Person>"], "<ZBW>": ["<Organisation>"], "<ORG2>": ["<Organisation>"]},
    expect_predicates_present=["<worksFor>", "<collabWith>"],
    expect_superedge_predicates_present=["<worksFor>", "<collabWith>"],
)

# Graph D: two data sources (zbw.eu and uni-kiel.de), same subject, two orgs (ZBW, CAU)
NQ_D = f"""\
<tbl> {RDF_TYPE} <Person> <http://zbw.eu> .
<tbl> <name> "Till Blume" <http://zbw.eu> .
<tbl> <worksFor> <ZBW> <http://zbw.eu> .
<tbl> <livesIn> "Kiel" <http://zbw.eu> .
<ZBW> {RDF_TYPE} <Organisation> <http://zbw.eu> .
<ZBW> <name> "Leibniz Information Centre for Economics" <http://zbw.eu> .
<tbl> {RDF_TYPE} <Person> <http://uni-kiel.de> .
<tbl> <name> "Till Blume" <http://uni-kiel.de> .
<tbl> <worksFor> <CAU> <http://uni-kiel.de> .
<tbl> <livesIn> "Kiel" <http://uni-kiel.de> .
<CAU> {RDF_TYPE} <Organisation> <http://uni-kiel.de> .
<CAU> <name> "University of Kiel" <http://uni-kiel.de> .
"""

print("\nGRAPH D — two non-literal edges (tbl→ZBW, tbl→CAU), two orgs, two sources")
run_unit_graph_test(
    NQ_D,
    include_literals=False,
    add_reverse_edges=False,
    strategy_cls=SEComplexAttributeClassCollection,  # captures dst types too
    expect_min_nodes=3,    # <tbl>, <ZBW>, <CAU>
    expect_min_edges=2,    # worksFor edges to ZBW and CAU
    expect_types_on_label={
        "<tbl>": ["<Person>"],
        "<ZBW>": ["<Organisation>"],
        "<CAU>": ["<Organisation>"],
    },
    expect_predicates_present=["<worksFor>"],
    expect_superedge_predicates_present=["<worksFor>"],
)

# Graph E: single source (uni-kiel.de), tbl worksFor CAU
NQ_E = f"""\
<tbl> {RDF_TYPE} <Person> <http://uni-kiel.de> .
<tbl> <name> "Till Blume" <http://uni-kiel.de> .
<tbl> <worksFor> <CAU> <http://uni-kiel.de> .
<tbl> <livesIn> "Kiel" <http://uni-kiel.de> .
<CAU> {RDF_TYPE} <Organisation> <http://uni-kiel.de> .
<CAU> <name> "University of Kiel" <http://uni-kiel.de> .
"""

print("\nGRAPH E — one non-literal edge (tbl→CAU), single org, single source")
run_unit_graph_test(
    NQ_E,
    include_literals=False,
    add_reverse_edges=False,
    strategy_cls=SEComplexAttributeClassCollection,
    expect_min_nodes=2,    # <tbl>, <CAU>
    expect_min_edges=1,    # tbl→CAU worksFor
    expect_types_on_label={
        "<tbl>": ["<Person>"],
        "<CAU>": ["<Organisation>"],
    },
    expect_predicates_present=["<worksFor>"],
    expect_superedge_predicates_present=["<worksFor>"],
)

# Graph F: three persons, only literal name attributes (no non-literal edges)
NQ_F = f"""\
<tbl> {RDF_TYPE} <Person> <http://zbw.eu> .
<tbl> <name> "Till Blume" <http://zbw.eu> .
<asc> {RDF_TYPE} <Person> <http://zbw.eu> .
<asc> <name> "Ansgar Scherp" <http://zbw.eu> .
<lro> {RDF_TYPE} <Person> <http://zbw.eu> .
<lro> <name> "Lars Rohwedder" <http://zbw.eu> .
"""

print("\nGRAPH F — three persons, only literal edges (names), so no non-literal edges")
run_unit_graph_test(
    NQ_F,
    include_literals=False,      # literals skipped → no edges
    add_reverse_edges=False,
    strategy_cls=SEClassCollection,  # class-only makes sense when there are no non-literal edges
    expect_min_nodes=3,          # <tbl>, <asc>, <lro>
    expect_min_edges=0,          # no non-literal relations
    expect_types_on_label={
        "<tbl>": ["<Person>"],
        "<asc>": ["<Person>"],
        "<lro>": ["<Person>"],
    },
    # no predicate/superedge checks here; there are no non-literal relations
)


Graph A — original graph + AttributeCollection
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
PASSED

Graph B — original graph + reverse edges (undirected-ish)
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
PASSED

Graph C — extended graph (two non-literal edges, two orgs)
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
Added type <Organisation> to node <ORG2>
PASSED

GRAPH D — two non-literal edges (tbl→ZBW, tbl→CAU), two orgs, two sources
Added type <Person> to node <tbl>
Added type <Organisation> to node <ZBW>
Added type <Person> to node <tbl>
Added type <Organisation> to node <CAU>
PASSED

GRAPH E — one non-literal edge (tbl→CAU), single org, single source
Added type <Person> to node <tbl>
Added type <Organisation> to node <CAU>
PASSED

GRAPH F — three persons, only literal edges (names), so no non-literal edges
Added type <Person> to node <tbl>
Added type <Person> to node <asc>
Added type <Person> to n

(<networkx.classes.multidigraph.MultiDiGraph at 0x24b333ecd50>,
 {'<tbl>': VertexSummary(id=234206306556079984568291513317827537541, types=1, preds=0),
  '<asc>': VertexSummary(id=234206306556079984568291513317827537541, types=1, preds=0),
  '<lro>': VertexSummary(id=234206306556079984568291513317827537541, types=1, preds=0)},
 {'<tbl>': 234206306556079984568291513317827537541,
  '<asc>': 234206306556079984568291513317827537541,
  '<lro>': 234206306556079984568291513317827537541})